# Where-function parsing and construction

Sometimes we define a function which constructs a PyTree from nodes of another PyTree. A typical case is a function which given a model PyTree, selects those nodes whose array leaves should be trainable. 

In [1]:
where_trainable = lambda model: (
    (
        model.hidden.layer1, 
        model.hidden.layer3,
    ),
    model.linear_out,
)

For example, `equinox.tree_at` uses such a "where-function" to select nodes of a PyTree to be updated out-of-place:

```python
updated_tree = equinox.tree_at(
    lambda tree: (tree.foo, tree.subtree.bar),
    tree_to_update,
    (new_foo, new_bar),
)
```

Importantly, while a where-function looks like a PyTree of data -- like a set of addresses we want to access -- it is *not* itself a PyTree. Being a function, it cannot be [serialised as a PyTree](https://docs.kidger.site/equinox/examples/serialisation/). And we probably shouldn't [pickle](https://docs.python.org/3/library/pickle.html) it. 

The cookbook provides a couple of functions that may be useful, here. The first is [`where_func_to_strs`][jax_cookbook.where_func_to_strs], which takes a where-function and returns its representation as a PyTree of strings:

In [7]:
from jax_cookbook import where_func_to_strs

where_trainable_strs = where_func_to_strs(where_trainable)

where_trainable_strs

(('hidden.layer1', 'hidden.layer3'), 'linear_out')

Similarly, `where_attr_strs_to_func` takes a PyTree of strings *representing attribute accesses*, and returns its representation as a where-function:

In [9]:
from jax_cookbook import where_attr_strs_to_func

where_trainable_parsed = where_attr_strs_to_func(where_trainable_strs)

type(where_trainable_parsed)

function

In [10]:
where_func_to_strs(where_trainable_parsed)

(('hidden.layer1', 'hidden.layer3'), 'linear_out')

Currently this only works for where-functions based on attribute accesses, such as those where-functions you'd typically use to select model parameters in a `equinox.Module`-based PyTree. However, maybe you have a where-function which refers to indices of sequences, or keys of dicts, and so on. If you only need to convert it to a string representation, then `where_func_to_labels` may work; but if you need to generate a working where-function from strings, the strings cannot contain indexing notation or other kinds of non-attribute access.